### RAG pipelines - Data Ingestion to Vector DB Pipel

In [19]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [20]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):

    all_documents = []
    pdf_dir = Path(pdf_directory)

    #Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\n Processing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load() 

            ### add Source information to metadata
            for doc in documents:
                doc.metadata['source_files'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error {e}")

    print(f"Total documents loaded {len(all_documents)}") 

    return all_documents  

all_pdf_documents = process_all_pdfs("../data/pdf")


Found 1 PDF files to process

 Processing: Rapport_Stage_Mandi_vf.pdf
 Loaded 35 pages
Total documents loaded 35


In [21]:
### Text Splitting get into chunks

def split_documents(documents, chunksize=1000, chunk_overlap= 200):
    """ Split documents into smaller chunks for better RAG performance """

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunksize,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n \n","\n"," ",""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #Show example of a chunk
    if split_docs:
        print(f"\n Example chunk:")
        print(f"Content: {split_docs[0].page_content[:200] } ...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [22]:
chunks = split_documents(all_pdf_documents)
chunks

Split 35 documents into 61 chunks

 Example chunk:
Content: Développement d’une plateforme SaaS intellig ente de gestion 
de la relation client basée sur l’intelligence artificielle 
 
 
 
 
Département Télécommunications, Réseaux & Informatique 
Ecole Nationa ...
Metadata: {'producer': 'pdfTeX-1.40.29', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-09-07T09:50:41+00:00', 'author': '', 'keywords': '', 'moddate': '2026-09-07T09:50:41+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Rapport_Stage_Mandi_vf.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1', 'source_files': 'Rapport_Stage_Mandi_vf.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.29', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-09-07T09:50:41+00:00', 'author': '', 'keywords': '', 'moddate': '2026-09-07T09:50:41+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Rapport_Stage_Mandi_vf.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1', 'source_files': 'Rapport_Stage_Mandi_vf.pdf', 'file_type': 'pdf'}, page_content='Développement d’une plateforme SaaS intellig ente de gestion \nde la relation client basée sur l’intelligence artificielle \n \n \n \n \nDépartement Télécommunications, Réseaux & Informatique \nEcole Nationale des Sciences Appliquées d’El Jadida \nUniversité Chouaib Doukkali              \n  \n \nRéf : 2ITE – F2-345 - 2026 \n \nRAPPORT DE STAGE II \n \n Ingénierie Informatique et Technologies Emergentes (2ITE) \n3ème année Cycle Ingénieur \n \n \n \n \nR

### embedding And vectorStoreDB

In [23]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [24]:
class EmbeddingManager:
    """ Handkes document embedding generation using SentenceTransformer"""

    def __init__(self, model_name:str="all-MiniLM-L6-V2"):
        """ Initialsize the embedding manager

        Args:
             model_name: HuggingFace model name for sentence embeddings
         
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""

        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")

        except Exception as e:
            print(f"Error loading model {self.model_name} : {e}")
            raise

    def generate_embedding(self,texts:List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        args:
            text: List of text strings to embed
        returns:
            numpy array of embeddings with shape (len(texts),embedding_dim) 
        """

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## initialize the embedding manager
Embedding_manager = EmbeddingManager()
Embedding_manager

    

Loading embedding model: all-MiniLM-L6-V2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5754.88it/s]


Model loaded successfully. Embedding dimension: 384


### Vector Store

In [25]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 61


In [26]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.29', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-09-07T09:50:41+00:00', 'author': '', 'keywords': '', 'moddate': '2026-09-07T09:50:41+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Rapport_Stage_Mandi_vf.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1', 'source_files': 'Rapport_Stage_Mandi_vf.pdf', 'file_type': 'pdf'}, page_content='Développement d’une plateforme SaaS intellig ente de gestion \nde la relation client basée sur l’intelligence artificielle \n \n \n \n \nDépartement Télécommunications, Réseaux & Informatique \nEcole Nationale des Sciences Appliquées d’El Jadida \nUniversité Chouaib Doukkali              \n  \n \nRéf : 2ITE – F2-345 - 2026 \n \nRAPPORT DE STAGE II \n \n Ingénierie Informatique et Technologies Emergentes (2ITE) \n3ème année Cycle Ingénieur \n \n \n \n \nR

In [27]:
###Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

###Generate embeddings
embeddings = Embedding_manager.generate_embedding(texts)

###Store in the vector database
vectorstore.add_documents(chunks, embeddings)


Generating embeddings for 61 texts...


Batches: 100%|██████████| 2/2 [00:05<00:00,  2.99s/it]

Generated embeddings with shape: (61, 384)
Adding 61 documents to vector store...
Successfully added 61 documents to vector store
Total documents in collection: 122


In [28]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store."""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        """
        Initialize the retriever.

        Args:
            vector_store: Vector store containing document embeddings.
            embedding_manager: Manager for generating query embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query.

        Args:
            query: The search query.
            top_k: Number of top results to return.
            score_threshold: Minimum similarity score threshold.

        Returns:
            A list of relevant documents with their scores and metadata.
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, threshold: {score_threshold}")

        # --------------------------------------------------
        # 1. Generate query embedding
        # --------------------------------------------------

        query_embedding = self.embedding_manager.generate_embedding([query])[0]

        # --------------------------------------------------
        # 2. Search in vector store
        # --------------------------------------------------

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # --------------------------------------------------
            # 3. Process results
            # --------------------------------------------------

            retrieved_docs = []

            if results["documents"] and results["documents"][0]:

                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                # --------------------------------------------------
                # 4. Calculate similarity and filter results
                # --------------------------------------------------

                for rank, (
                    doc_id,
                    document,
                    metadata,
                    distance
                ) in enumerate(
                    zip(ids, documents, metadatas, distances),
                    start=1
                ):

                    # ChromaDB returns cosine distance.
                    # Convert distance to similarity score.
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:

                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": rank
                        })

                print(
                    f"Retrieved {len(retrieved_docs)} documents "
                    f"(after filtering)"
                )

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retiever = RAGRetriever(vectorstore, Embedding_manager)

In [29]:
rag_retiever.retrieve("c est quoi pig")

Retrieving documents for query: 'c est quoi pig'
Top K: 5, threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 21.61it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

### Integration Vectordb context pipeline with LLM output

In [30]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv()

# Initialize the OpenAI LLM
openai_api_key = os.getenv("OPEN_AI_KEY")

llm = ChatOpenAI(
    api_key = openai_api_key,
    model = "gpt-4o-mini",
    temperature = 0.1,
    max_tokens = 1024
)

# Simple RAG function: retriever context + generate response

def rag_simple(query, retriever, llm, top_k=3):
    #Retrieve relevant context 
    results = retriever.retrieve(
        query,
        top_k = top_k
    )

    #Combine retrieved documents
    context = "\n \n".join(
        [ doc["content"] for doc in results]
    ) if results else ""

    #No context found
    if not context:
        return "No relevant context found to answer the question"

    #Generate prompt
    prompt = f""" 
    Use the following context to answer the question consicely. Only use information from the provided context.
    
    Context:{context} 
    
    Question: {query}
    
    Answer:
    """

    #generate answer using OpenAI
    response = llm.invoke(prompt)
    return response.content

    

In [37]:
answer = rag_simple("c'est quoi une platforme SAAS ? ",rag_retiever,llm)
print(answer)

Retrieving documents for query: 'c'est quoi une platforme SAAS ? '
Top K: 3, threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 99.82it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


Une plateforme SaaS (Software as a Service) est un modèle de distribution de logiciels où l'application est hébergée par un fournisseur et mise à disposition des utilisateurs via Internet, généralement sous forme d'abonnement.


### Enhanced RAG pipeline Features

In [36]:
def rag_advanced(query, retriever,llm, top_k= 5, min_score=0.2, return_context=False):
    """RAG pipline with extra features
    - Returns answer, sources, confidence score and optionally full context
    """

    results = retriever.retrieve(query, top_k=top_k, score_threshold = min_score)
    if not results:
        return {'answer' : 'No relevant context found','sources':[],'confidence':0.0,'context':''}

    #Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unkown')),
        'page': doc['metadata'].get('page','unkown'),
        'score' : doc['similarity_score'],
        'preview' : doc['content'][:300] + '...'
    }for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    #Generate answer
    prompt = f""" Use the following context to answer the question concisely. \n Context:\n  \n Question: {query} \n \n Answer:"""
    response = llm.invoke([prompt.format(context = context, query = query)])

    output = {
        'answer' : response.content,
        'sources': sources,
        'confidence' : confidence
    }

    if return_context:
        output['context'] = context

    return output

#Exemple 
result = rag_advanced("c'est quoi le customer success  ? ", rag_retiever,llm,top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview", result['context'][:300])



Retrieving documents for query: 'c'est quoi le customer success  ? '
Top K: 3, threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 62.99it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


Answer: Le customer success, ou succès client, est une approche stratégique visant à garantir que les clients tirent le maximum de valeur de produits ou services, favorisant ainsi leur satisfaction, leur fidélité et leur rétention. Cela implique un suivi proactif des besoins des clients, une communication régulière et un soutien pour les aider à atteindre leurs objectifs.
sources: [{'source': '..\\data\\pdf\\Rapport_Stage_Mandi_vf.pdf', 'page': 11, 'score': 0.15231066942214966, 'preview': 'Le Customer Success est une approche stratégique visant à accompagner les clients tout au\nlong de leur parcours avec l’entreprise, afin de s’assurer qu’ils tirent pleinement valeur du produit\nou service acquis. Contrairement au support classique (réactif), le Customer Success est proactif :\nil antic...'}, {'source': '..\\data\\pdf\\Rapport_Stage_Mandi_vf.pdf', 'page': 11, 'score': 0.15231066942214966, 'preview': 'Le Customer Success est une approche stratégique visant à accompagner les clients tou

In [33]:
print(chunks)

[Document(metadata={'producer': 'pdfTeX-1.40.29', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-09-07T09:50:41+00:00', 'author': '', 'keywords': '', 'moddate': '2026-09-07T09:50:41+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Rapport_Stage_Mandi_vf.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1', 'source_files': 'Rapport_Stage_Mandi_vf.pdf', 'file_type': 'pdf'}, page_content='Développement d’une plateforme SaaS intellig ente de gestion \nde la relation client basée sur l’intelligence artificielle \n \n \n \n \nDépartement Télécommunications, Réseaux & Informatique \nEcole Nationale des Sciences Appliquées d’El Jadida \nUniversité Chouaib Doukkali              \n  \n \nRéf : 2ITE – F2-345 - 2026 \n \nRAPPORT DE STAGE II \n \n Ingénierie Informatique et Technologies Emergentes (2ITE) \n3ème année Cycle Ingénieur \n \n \n \n \nR